<a href="https://colab.research.google.com/github/JosephBigDataAnalytics/JKaremera-Programming-BigDataAnalytics/blob/main/Task_16_complete_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Required Task 16
Add a new tool called get_pe_ratio that fetches the Price-to-Earnings ratio. Then ask the agent if Apple is 'overvalued' compared to the average market P/E of 25.

Hint:
stock = yf.Ticker(ticker)
# PE ratio is often in 'summaryDetail' or 'info'
 pe = stock.info.get('trailingPE', 'N/A')


Custom Tools (Python Functions)

get_stock_price(ticker)

get_company_risk_score (ticker)

get_pe_ratio

Setup and API configuration

In [ ]:
# 1. Setup (Use the standard library)
# Get your API key from https://aistudio.google.com/
!pip install -q -U google-generativeai # Ensure the correct library is installed
import google.generativeai as genai # Import the correct module to override previous
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('gemniapi')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Install Yfinance library

In [ ]:
!pip install -q yfinance

define tool functions:

get_stock_price(ticker)

get_company_risk_score (ticker)

get_pe_ratio

1. get_stock_price(ticker): Takes a stock ticker symbol (e.g., 'AAPL') and returns its current live price.

2. get_company_risk_score(ticker): Calculates a simplified risk assessment for a given stock based on its Beta value, which measures market volatility. It categorizes risk as 'High', 'Moderate', or 'Low'.

3. get_pe_ratio(ticker): fetches the Price-to-Earnings ratio. Then ask the agent if Apple is 'overvalued' compared to the average market P/E of 25.

In [ ]:
import yfinance as yf

# Define the "Real" Tools

def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        # .fast_info is much faster than .info for just price
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"

def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... ⚠️ TOOL CALL: Fetching Risk Metrics for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)

        # We can add some logic to make it more "human-readable" for the agent
        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"

        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"

def get_pe_ratio(ticker: str):
    """
    Fetches the Price-to-Earnings ratio for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL').
    """
    print(f"  ... 📊 TOOL CALL: Fetching P/E Ratio for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', 'N/A')
        return pe
    except Exception as e:
        return f"Error fetching P/E ratio for {ticker}: {e}"





print("✅ Real-Time Tools Defined.")

✅ Real-Time Tools Defined.


Initialize the Generative Model with Tools

In [ ]:
# 4. Initialize Model with Tools
# We use 'gemini-2.5-flash' as it is fast and excellent at tool use.

tools_list = [get_stock_price, get_company_risk_score,get_pe_ratio]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

# Enable automatic function calling
# This tells the SDK: "If the model asks to run a function, just run it for me and give the answer back."
chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Model initialized with Agentic capabilities.")

✅ Model initialized with Agentic capabilities.


Test: single tOOL, call

In [ ]:
response = chat.send_message("Is Apple overvalued compared to the average market P/E of 25?")

print("\n🤖 AGENT RESPONSE:")
print(response.text)

  ... 📊 TOOL CALL: Fetching P/E Ratio for AAPL ...

🤖 AGENT RESPONSE:
Apple's P/E ratio is 31.66, which is higher than the average market P/E of 25. This suggests that Apple might be overvalued compared to the average market.
